# CourtLens — full local pipeline on Kaggle GPU
Same path as local: YOLO+ByteTrack (stride 5) → kit/jersey identity → possession events → match_stats.
Requires Accelerator = **GPU T4** in notebook settings.

In [ ]:
import glob, subprocess, sys, zipfile, shutil, os
from pathlib import Path

root = Path("/kaggle/input")
print("INPUT TREE:")
for p in sorted(root.rglob("*"))[:80]:
    if p.is_file():
        print(" ", p, p.stat().st_size)

wheels = sorted(glob.glob(str(root / "**/*.whl"), recursive=True))
assert wheels, "No wheels in dataset"
print("wheels", len(wheels))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *wheels])

# CourtLens package: directory zipped by Kaggle as courtlens_offline.zip, or loose .py
pkg_dir = Path("/kaggle/working/courtlens_offline")
if pkg_dir.exists():
    shutil.rmtree(pkg_dir)
pkg_dir.mkdir(parents=True)

zips = sorted(glob.glob(str(root / "**/courtlens_offline.zip"), recursive=True))
zips += sorted(glob.glob(str(root / "**/*courtlens*.zip"), recursive=True))
py_runs = sorted(glob.glob(str(root / "**/run.py"), recursive=True))
print("zips", zips)
print("run.py", py_runs)

if zips:
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(pkg_dir)
elif py_runs:
    src = Path(py_runs[0]).parent
    for f in src.glob("*.py"):
        shutil.copy2(f, pkg_dir / f.name)
else:
    raise SystemExit("courtlens_offline package not found in dataset input")

if not (pkg_dir / "run.py").exists():
    nested = list(pkg_dir.glob("**/run.py"))
    assert nested, f"run.py missing after extract; contents={list(pkg_dir.rglob('*'))}"
    for f in nested[0].parent.glob("*.py"):
        shutil.copy2(f, pkg_dir / f.name)

# Hard requirement: VisionBackend must exist
vis = pkg_dir / "vision.py"
text = vis.read_text() if vis.exists() else ""
if "class VisionBackend" not in text:
    vis.write_text(
        "from __future__ import annotations\n"
        "from abc import ABC, abstractmethod\n"
        "from dataclasses import dataclass\n"
        "from typing import Any\n\n"
        "@dataclass\n"
        "class Detection:\n"
        "    track_key: str\n"
        "    camera_label: str\n"
        "    start_ms: int\n"
        "    end_ms: int\n"
        "    confidence: float\n"
        "    bbox_series: list[dict[str, Any]] | None = None\n"
        "    class_name: str = 'person'\n"
        "    features: dict[str, Any] | None = None\n\n"
        "@dataclass\n"
        "class EventCandidate:\n"
        "    event_type: str\n"
        "    camera_label: str\n"
        "    camera_ts_ms: int\n"
        "    confidence: float\n"
        "    track_keys: list[str]\n"
        "    evidence: dict[str, Any] | None = None\n\n"
        "class VisionBackend(ABC):\n"
        "    name: str\n"
        "    version: str\n"
        "    last_run_stats: dict[str, Any]\n"
        "    @abstractmethod\n"
        "    def detect_and_track(self, video_path: str, camera_label: str):\n"
        "        raise NotImplementedError\n"
        "    @abstractmethod\n"
        "    def infer_events(self, detections, camera_label: str):\n"
        "        raise NotImplementedError\n"
    )
    print("wrote VisionBackend fallback into vision.py")
else:
    print("vision.py already has VisionBackend")

print("package files", sorted(p.name for p in pkg_dir.glob("*.py")))
subprocess.call(["bash", "-lc", "tesseract --version | head -1 || echo 'no tesseract binary'"])
print("install done")


In [ ]:
import sys, torch, shutil, subprocess
print("torch", torch.__version__)
cuda = torch.cuda.is_available()
print("cuda available", cuda)
if cuda:
    print("device", torch.cuda.get_device_name(0))
else:
    smi = shutil.which("nvidia-smi")
    if smi:
        subprocess.call([smi])
    else:
        print("nvidia-smi not present (CPU session)")
    print("FATAL: GPU required. In Kaggle UI: Settings → Accelerator → GPU T4, then re-run.")
    sys.exit(1)


In [ ]:
import glob, json, os, sys, traceback
from pathlib import Path

pkg = Path("/kaggle/working/courtlens_offline")
if not (pkg / "run.py").exists():
    pkg = next(Path("/kaggle/working").glob("**/run.py")).parent
sys.path.insert(0, str(pkg))

weights = glob.glob("/kaggle/input/**/yolov8n.pt", recursive=True)[0]
# Full testing_videos corpus (short names on the dataset)
CORPUS = [
    "test_video_2.mp4",  # facility CCTV — product gate
    "portugal_spain_euro_2026_final.mp4",
    "bangladesh_pakistan_saff_2026.mp4",
    "spain_brazil_fifa_2012.mp4",
    "spain_france_u19_2023.mp4",
]
all_mp4 = {Path(p).name: p for p in glob.glob("/kaggle/input/**/*.mp4", recursive=True)}
print("available mp4", sorted(all_mp4))
missing = [n for n in CORPUS if n not in all_mp4]
if missing:
    print("WARNING missing", missing)

os.environ["COURTLENS_WEIGHTS"] = weights
os.environ["VISION_FRAME_STRIDE"] = "5"
os.environ["VISION_JERSEY_OCR"] = "1"
os.environ["VISION_OCR_TIMEOUT_S"] = "0.8"
print("weights", weights)

import run as courtlens_run

index = []
root_out = Path("/kaggle/working/corpus_out")
root_out.mkdir(exist_ok=True)
for name in CORPUS:
    if name not in all_mp4:
        index.append({"video": name, "error": "missing"})
        continue
    slug = Path(name).stem
    out = root_out / slug
    os.environ["COURTLENS_VIDEO"] = all_mp4[name]
    os.environ["COURTLENS_OUT"] = str(out)
    print("=" * 72)
    print("VIDEO", name, "→", out)
    try:
        courtlens_run.main()
        table = json.loads((out / "player_table.json").read_text()) if (out / "player_table.json").exists() else []
        stats = json.loads((out / "match_stats.json").read_text()) if (out / "match_stats.json").exists() else {}
        index.append({
            "video": name,
            "slug": slug,
            "out": str(out),
            "totals": stats.get("totals") or {},
            "players": len(table),
            "detection": stats.get("detection") or {},
        })
    except Exception as e:
        traceback.print_exc()
        index.append({"video": name, "slug": slug, "error": str(e)})

(root_out / "corpus_index.json").write_text(json.dumps(index, indent=2))
print("CORPUS DONE")
print(json.dumps(index, indent=2))
